# Bayesian Inference

## Learning Objectives
1. Perform Beta-Binomial conjugate Bayesian updates in numpy and scipy
2. Implement Gaussian-Gaussian conjugate update for mean estimation
3. Apply Bayesian A/B testing with daily sequential updates
4. Compare MAP vs MLE vs Full Bayesian on regression: bias, variance, uncertainty

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from scipy.special import betaln

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Imports OK')
print('NumPy:', np.__version__)

## Level 1: Beta-Binomial Conjugate Update

The Beta distribution is the conjugate prior for the Bernoulli/Binomial likelihood.

**Update rule**: Prior Beta(alpha, beta) + n trials with k successes
=> Posterior Beta(alpha + k, beta + n - k)

This is analytical — no numerical integration needed.

In [ ]:
# --- Setup: estimating a coin's bias ---
# We flip a coin 10 times. Observe 7 heads.
# Prior belief: Beta(1, 1) = uniform (could be any bias)
n_trials  = 10
k_heads   = 7
alpha_pri = 1.0   # prior Beta(1, 1)
beta_pri  = 1.0

# Posterior update (conjugate)
alpha_post = alpha_pri + k_heads
beta_post  = beta_pri  + (n_trials - k_heads)
print(f'Prior:     Beta({alpha_pri:.0f}, {beta_pri:.0f})')
print(f'Data:      {k_heads} heads in {n_trials} flips')
print(f'Posterior: Beta({alpha_post:.0f}, {beta_post:.0f})')
print(f'Posterior mean: {alpha_post/(alpha_post+beta_post):.4f}'
      f'  (MLE would give {k_heads/n_trials:.4f})')

# --- Plot prior, likelihood, posterior ---
theta = np.linspace(0, 1, 500)

prior_dist  = stats.beta(alpha_pri,  beta_pri)
post_dist   = stats.beta(alpha_post, beta_post)

# Likelihood: proportional to theta^k * (1-theta)^(n-k)
# Normalize for plotting
likelihood_unnorm = theta ** k_heads * (1 - theta) ** (n_trials - k_heads)
likelihood_norm   = likelihood_unnorm / np.trapz(likelihood_unnorm, theta)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Prior / likelihood / posterior
axes[0].plot(theta, prior_dist.pdf(theta), 'g-', lw=2, label=f'Prior Beta({alpha_pri:.0f},{beta_pri:.0f})')
axes[0].plot(theta, likelihood_norm, 'b-', lw=2, label=f'Likelihood (7H,3T normalized)')
axes[0].plot(theta, post_dist.pdf(theta), 'r-', lw=2, label=f'Posterior Beta({alpha_post:.0f},{beta_post:.0f})')
axes[0].axvline(k_heads/n_trials, color='b', ls=':', label=f'MLE={k_heads/n_trials:.2f}')
axes[0].axvline(post_dist.mean(), color='r', ls=':', label=f'Post.mean={post_dist.mean():.3f}')
axes[0].set_xlabel('theta (coin bias)')
axes[0].set_ylabel('Density')
axes[0].set_title('Bayesian Update: Beta-Binomial', fontweight='bold')
axes[0].legend(fontsize=9)

# Sequential update as data accumulates
np.random.seed(42)
true_p = 0.65
n_seq  = 50
flips  = np.random.binomial(1, true_p, n_seq)

a, b = 1.0, 1.0
posterior_means_seq = []
ci_los, ci_his = [], []
for flip in flips:
    a += flip
    b += 1 - flip
    d = stats.beta(a, b)
    posterior_means_seq.append(d.mean())
    ci_los.append(d.ppf(0.025))
    ci_his.append(d.ppf(0.975))

n_arr = np.arange(1, n_seq + 1)
axes[1].plot(n_arr, posterior_means_seq, 'r-', lw=2, label='Posterior mean')
axes[1].fill_between(n_arr, ci_los, ci_his, alpha=0.2, color='red', label='95% credible interval')
axes[1].axhline(true_p, color='black', ls='--', lw=2, label=f'True p={true_p}')
axes[1].set_xlabel('Number of flips')
axes[1].set_ylabel('Estimated bias')
axes[1].set_title('Sequential Bayesian Update', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('stats_03_beta_binomial.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_03_beta_binomial.png')

# --- Effect of prior strength (informative vs uninformative) ---
print('\nEffect of prior on posterior with 10 observations (7 heads):')
prior_configs = [(1,1,'Weak uniform'), (5,5,'Moderate (prior: 50/50)'), (20,20,'Strong (prior: 50/50)')]
for (a0, b0, name) in prior_configs:
    post = stats.beta(a0 + k_heads, b0 + n_trials - k_heads)
    ci   = post.ppf([0.025, 0.975])
    print(f'  {name}: posterior mean={post.mean():.3f}  95% CI=[{ci[0]:.3f}, {ci[1]:.3f}]')

## Level 2: Gaussian-Gaussian Conjugate Update

When the likelihood is Gaussian with known variance and the prior is also Gaussian,
the posterior is Gaussian with analytically computed mean and variance.

This shows how the posterior mean is a precision-weighted average of
the prior mean and the sample mean.

In [ ]:
# --- Gaussian-Gaussian conjugate update ---
# Model: X_i ~ N(mu, sigma_known^2)
# Prior: mu ~ N(mu_0, sigma_0^2)
# Posterior: mu | data ~ N(mu_n, sigma_n^2)
#   sigma_n^2 = 1 / (1/sigma_0^2 + n/sigma_known^2)
#   mu_n = sigma_n^2 * (mu_0/sigma_0^2 + n*x_bar/sigma_known^2)

def gaussian_conjugate_update(mu_0, sigma_0, sigma_known, data):
    # Precision = 1/variance
    tau_0     = 1.0 / sigma_0**2      # prior precision
    tau_known = 1.0 / sigma_known**2  # data precision per observation
    n    = len(data)
    xbar = data.mean()

    # Posterior precision = prior precision + n * data precision
    tau_n = tau_0 + n * tau_known
    sigma_n = 1.0 / np.sqrt(tau_n)

    # Posterior mean = precision-weighted average
    mu_n = (tau_0 * mu_0 + n * tau_known * xbar) / tau_n

    return mu_n, sigma_n

# True parameter
true_mu = 5.0
sigma_known = 2.0   # known noise std

# Prior: slightly wrong (mu_0=3), moderately uncertain (sigma_0=1.5)
mu_0    = 3.0
sigma_0 = 1.5

# Show how posterior updates as n grows
sample_sizes_test = [1, 5, 10, 20, 50, 100, 500]
data_full = np.random.normal(true_mu, sigma_known, 500)

print('Gaussian-Gaussian conjugate update:')
print(f'  True mu={true_mu}, sigma_known={sigma_known}')
print(f'  Prior: N({mu_0}, {sigma_0}^2)')
print(f'  {'n':<6} {'Post.mean':>12} {'Post.std':>10} {'95% CI':>25}')
print('  ' + '-' * 56)

post_means, post_stds = [], []
for n in sample_sizes_test:
    data_n = data_full[:n]
    mu_n, sig_n = gaussian_conjugate_update(mu_0, sigma_0, sigma_known, data_n)
    ci_lo = mu_n - 1.96 * sig_n
    ci_hi = mu_n + 1.96 * sig_n
    post_means.append(mu_n)
    post_stds.append(sig_n)
    print(f'  {n:<6} {mu_n:>12.4f} {sig_n:>10.4f} [{ci_lo:.3f}, {ci_hi:.3f}]')

# --- Prior predictive vs posterior predictive ---
x_range = np.linspace(-4, 12, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Posterior distributions for different n values
colors = plt.cm.Blues(np.linspace(0.3, 1.0, len(sample_sizes_test)))
axes[0].plot(x_range, stats.norm.pdf(x_range, mu_0, sigma_0),
             'g-', lw=2, label=f'Prior N({mu_0},{sigma_0})')
for n, mu_n, sig_n, c in zip(sample_sizes_test, post_means, post_stds, colors):
    axes[0].plot(x_range, stats.norm.pdf(x_range, mu_n, sig_n),
                 color=c, lw=1.5, alpha=0.8, label=f'n={n}')
axes[0].axvline(true_mu, color='red', ls='--', lw=2, label=f'True mu={true_mu}')
axes[0].set_xlabel('mu'); axes[0].set_ylabel('Density')
axes[0].set_title('Posterior Narrows With More Data', fontweight='bold')
axes[0].legend(fontsize=7, ncol=2)

# Posterior std vs n (compare: prior std, posterior std, frequentist SE = sigma/sqrt(n))
n_arr = np.array(sample_sizes_test)
freq_se = sigma_known / np.sqrt(n_arr)  # frequentist standard error
axes[1].semilogx(n_arr, post_stds, 'b-o', lw=2, label='Posterior std')
axes[1].semilogx(n_arr, freq_se, 'r--s', lw=2, label='Freq SE = sigma/sqrt(n)')
axes[1].axhline(sigma_0, color='green', ls=':', label=f'Prior std={sigma_0}')
axes[1].set_xlabel('n (log scale)'); axes[1].set_ylabel('Uncertainty')
axes[1].set_title('Bayesian vs Frequentist Uncertainty', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('stats_03_gaussian.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_03_gaussian.png')

print('\nKey insight: posterior precision = prior precision + n * data precision')
print('  As n grows, data precision dominates and prior becomes irrelevant')

## Real-World Example 1: Bayesian A/B Test (7-Day Sequential Update)

We run a conversion rate A/B test for 7 days, updating Beta posteriors each day.
We compute P(B > A) daily and track when we have enough evidence to decide.

Key advantage over frequentist: no need to pre-specify sample size;
we can make probabilistic decisions at any point.

In [ ]:
np.random.seed(42)

# True conversion rates
true_A = 0.10
true_B = 0.12
n_per_day = 300   # visitors per variant per day
n_days = 7

# Daily data simulation
conv_A = np.random.binomial(n_per_day, true_A, n_days)
conv_B = np.random.binomial(n_per_day, true_B, n_days)

print('Daily conversion data:')
print(f'  {'Day':<5} {'A conv/n':>10} {'B conv/n':>10} {'P(B>A)':>10} {'Decision'}' )
print('  ' + '-' * 55)

# Start with Beta(1,1) uninformative priors
aA, bA = 1.0, 1.0
aB, bB = 1.0, 1.0
n_mc   = 50_000

daily_pb_gt_pa = []
daily_lift_mean = []
daily_ci_lo = []
daily_ci_hi = []

for day in range(n_days):
    # Update posteriors
    aA += conv_A[day]; bA += n_per_day - conv_A[day]
    aB += conv_B[day]; bB += n_per_day - conv_B[day]

    # Monte Carlo: P(B > A) from posterior samples
    sampA = stats.beta(aA, bA).rvs(n_mc, random_state=day)
    sampB = stats.beta(aB, bB).rvs(n_mc, random_state=day+100)
    lift  = sampB - sampA

    pb_gt_pa = float(np.mean(lift > 0))
    lift_mean = float(lift.mean())
    ci_lo = float(np.percentile(lift, 2.5))
    ci_hi = float(np.percentile(lift, 97.5))

    daily_pb_gt_pa.append(pb_gt_pa)
    daily_lift_mean.append(lift_mean)
    daily_ci_lo.append(ci_lo)
    daily_ci_hi.append(ci_hi)

    cum_nA = int((day+1) * n_per_day)
    cum_nB = int((day+1) * n_per_day)
    cum_kA = int(conv_A[:day+1].sum())
    cum_kB = int(conv_B[:day+1].sum())

    if pb_gt_pa >= 0.95:
        decision = 'SHIP B'
    elif pb_gt_pa <= 0.05:
        decision = 'KEEP A'
    else:
        decision = 'Continue'

    print(f'  Day {day+1:<2}  {cum_kA}/{cum_nA:<5}    {cum_kB}/{cum_nB:<5}   {pb_gt_pa:.3f}   {decision}')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
days = np.arange(1, n_days + 1)

axes[0].plot(days, daily_pb_gt_pa, 'b-o', lw=2, markersize=7, label='P(B > A | data)')
axes[0].axhline(0.95, color='green', ls='--', lw=1.5, label='Ship threshold (0.95)')
axes[0].axhline(0.50, color='gray',  ls=':',  lw=1.5, label='No information (0.50)')
axes[0].set_xlabel('Day'); axes[0].set_ylabel('P(B > A | data)')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Bayesian A/B Test: Daily P(B > A)', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(days, [v*100 for v in daily_lift_mean], 'b-o', lw=2, label='Mean lift')
axes[1].fill_between(days,
                     [v*100 for v in daily_ci_lo],
                     [v*100 for v in daily_ci_hi],
                     alpha=0.2, color='blue', label='95% credible interval')
axes[1].axhline(0, color='red', ls='--', lw=1.5, label='No lift')
axes[1].axhline((true_B - true_A)*100, color='black', ls=':', label=f'True lift={(true_B-true_A)*100:.1f}%')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Lift (B rate - A rate) %')
axes[1].set_title('Posterior Lift Estimate Over Time', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('stats_03_ab_test.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_03_ab_test.png')

## Real-World Example 2: MAP vs MLE on Linear Regression

MAP with a Gaussian prior on weights = Ridge regression (L2 regularization).
MAP with a Laplace prior = Lasso regression (L1 regularization).

We demonstrate on an overfit scenario: more features than samples.
MLE overfits; MAP with Gaussian prior regularizes and generalizes better.

In [ ]:
np.random.seed(42)

# --- Small dataset: n=30 samples, d=25 features (overfit regime) ---
n_train = 30
n_test  = 500
d = 25  # feature dimension

# True sparse weights: only first 5 features matter
w_true = np.zeros(d)
w_true[:5] = [3.0, -2.0, 1.5, -1.0, 0.5]

# Generate data
X_train = np.random.randn(n_train, d)
X_test  = np.random.randn(n_test,  d)
noise = 0.5
y_train = X_train @ w_true + np.random.randn(n_train) * noise
y_test  = X_test  @ w_true + np.random.randn(n_test)  * noise

# --- MLE (ordinary least squares): analytical solution ---
# w_MLE = (X^T X)^{-1} X^T y
XtX = X_train.T @ X_train
Xty = X_train.T @ y_train
try:
    w_mle = np.linalg.solve(XtX, Xty)
except np.linalg.LinAlgError:
    w_mle = np.linalg.lstsq(X_train, y_train, rcond=None)[0]

# --- MAP with Gaussian prior (Ridge): w_MAP = (X^T X + lambda*I)^{-1} X^T y ---
# lambda = sigma^2 / sigma_prior^2 = noise variance / prior variance
def ridge_map(X_tr, y_tr, lam):
    d_ = X_tr.shape[1]
    XtX_ = X_tr.T @ X_tr + lam * np.eye(d_)
    Xty_ = X_tr.T @ y_tr
    return np.linalg.solve(XtX_, Xty_)

# Try several regularization strengths
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0]
mle_train_rmse = np.sqrt(np.mean((X_train @ w_mle - y_train)**2))
mle_test_rmse  = np.sqrt(np.mean((X_test  @ w_mle - y_test)**2))
print(f'MLE: train RMSE={mle_train_rmse:.3f}, test RMSE={mle_test_rmse:.3f} (overfit)')

print(f'\nMAP (Ridge) with different lambda (Gaussian prior strength):')
print(f'  {'lambda':<10} {'Train RMSE':>12} {'Test RMSE':>12} {'||w||':>10}')
print('  ' + '-' * 48)

best_lam, best_test = None, np.inf
map_weights, map_test_rmses = [], []
for lam in lambdas:
    w_map = ridge_map(X_train, y_train, lam)
    tr = np.sqrt(np.mean((X_train @ w_map - y_train)**2))
    te = np.sqrt(np.mean((X_test  @ w_map - y_test)**2))
    wnorm = np.linalg.norm(w_map)
    map_weights.append(w_map)
    map_test_rmses.append(te)
    if te < best_test:
        best_test, best_lam = te, lam
    print(f'  {lam:<10} {tr:>12.3f} {te:>12.3f} {wnorm:>10.3f}')

print(f'\nBest lambda={best_lam}: test RMSE={best_test:.3f}')
print(f'MAP reduces test RMSE by {(mle_test_rmse-best_test)/mle_test_rmse*100:.1f}% vs MLE')

# --- Plot: weight profiles ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
feat_idx = np.arange(d)
best_w = map_weights[lambdas.index(best_lam)]

axes[0].bar(feat_idx - 0.2, w_true, width=0.4, color='black', alpha=0.7, label='True w')
axes[0].bar(feat_idx + 0.2, w_mle,  width=0.4, color='red',   alpha=0.5, label='MLE w (overfit)')
axes[0].set_xlabel('Feature index'); axes[0].set_ylabel('Weight')
axes[0].set_title('Weight Profiles: True vs MLE', fontweight='bold')
axes[0].legend()

axes[1].bar(feat_idx - 0.2, w_true,  width=0.4, color='black', alpha=0.7, label='True w')
axes[1].bar(feat_idx + 0.2, best_w,  width=0.4, color='blue',  alpha=0.5, label=f'MAP w (lambda={best_lam})')
axes[1].set_xlabel('Feature index'); axes[1].set_ylabel('Weight')
axes[1].set_title('Weight Profiles: True vs MAP (Ridge)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('stats_03_map_vs_mle.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_03_map_vs_mle.png')

## Real-World Example 3: Hierarchical Model -- Sharing Information Across Groups

We estimate click-through rates for 10 ads that share a common (unknown) mean.
A hierarchical Bayesian model borrows strength across ads:
ads with little data are shrunk toward the group mean, reducing variance.

In [ ]:
np.random.seed(42)

# --- 10 ads with different click-through rates ---
# True CTRs drawn from a Beta(5, 45) group distribution (mean ~10%)
n_ads = 10
group_alpha = 5.0
group_beta  = 45.0
true_ctrs   = stats.beta(group_alpha, group_beta).rvs(n_ads, random_state=0)

# Each ad has different amounts of data
impressions = np.array([10, 20, 50, 100, 200, 10, 30, 15, 500, 5])
clicks      = np.array([int(ctr * n) for ctr, n in zip(true_ctrs, impressions)])

print('Ad-level data:')
print(f'  {'Ad':<5} {'Impressions':>12} {'Clicks':>8} {'Observed CTR':>14} {'True CTR':>10}')
print('  ' + '-' * 54)
for i in range(n_ads):
    obs_rate = clicks[i] / impressions[i] if impressions[i] > 0 else 0.0
    print(f'  Ad{i+1:<3} {impressions[i]:>12} {clicks[i]:>8} {obs_rate*100:>12.1f}%  {true_ctrs[i]*100:>8.1f}%')

# --- Method 1: MLE per ad (no pooling) ---
mle_ctrs = clicks / impressions

# --- Method 2: Bayesian update with uninformative prior ---
# Beta(1,1) + data -> Beta(1+k, 1+n-k)
bayes_ctrs = (1 + clicks) / (2 + impressions)

# --- Method 3: Hierarchical (empirical Bayes) ---
# Estimate group hyperparameters from data, then use as informative prior
# Group prior: Beta(alpha_est, beta_est) estimated from all ads
# Approximate: method of moments on observed rates
obs_mean = mle_ctrs.mean()
obs_var  = mle_ctrs.var()

# Method of moments for Beta parameters
# E[X] = a/(a+b), Var[X] = ab/((a+b)^2 (a+b+1))
def beta_mom(mean, var):
    if var >= mean * (1 - mean):
        return 1.0, 1.0  # fallback: uniform
    common = mean * (1 - mean) / var - 1
    return mean * common, (1 - mean) * common

est_alpha, est_beta = beta_mom(obs_mean, obs_var)
print(f'\nEstimated group prior: Beta({est_alpha:.2f}, {est_beta:.2f})'
      f' (mean={est_alpha/(est_alpha+est_beta)*100:.1f}%)')

# Posterior per ad using estimated group prior
hier_alpha = est_alpha + clicks
hier_beta  = est_beta  + impressions - clicks
hier_ctrs  = hier_alpha / (hier_alpha + hier_beta)  # posterior means

# --- Comparison ---
print('\nEstimation comparison:')
print(f'  {'Ad':<5} {'True':>8} {'MLE':>8} {'Bayesian':>10} {'Hierarchical':>14} {'Hier.Shrinkage'}')
print('  ' + '-' * 65)
for i in range(n_ads):
    shrinkage = hier_ctrs[i] - mle_ctrs[i]
    print(f'  Ad{i+1:<3} {true_ctrs[i]*100:>7.1f}% {mle_ctrs[i]*100:>7.1f}% {bayes_ctrs[i]*100:>9.1f}% {hier_ctrs[i]*100:>13.1f}%  {shrinkage*100:+.1f}%')

# Compare MSE
mse_mle  = np.mean((mle_ctrs - true_ctrs)**2)
mse_bayes = np.mean((bayes_ctrs - true_ctrs)**2)
mse_hier  = np.mean((hier_ctrs - true_ctrs)**2)
print(f'\nMSE comparison:')
print(f'  MLE:          {mse_mle*10000:.2f}  (x10^-4)')
print(f'  Bayesian(1,1): {mse_bayes*10000:.2f}  (x10^-4)')
print(f'  Hierarchical:  {mse_hier*10000:.2f}  (x10^-4)  <- lowest with shared prior')

# --- Plot comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
ads = np.arange(n_ads)
ax.scatter(ads, true_ctrs * 100, color='black', s=100, zorder=5, label='True CTR', marker='*')
ax.scatter(ads - 0.2, mle_ctrs  * 100, color='red',    s=60,  marker='o', label='MLE')
ax.scatter(ads,        bayes_ctrs* 100, color='green',  s=60,  marker='s', label='Bayesian Beta(1,1)')
ax.scatter(ads + 0.2, hier_ctrs  * 100, color='blue',   s=60,  marker='^', label='Hierarchical')
ax.set_xticks(ads)
ax.set_xticklabels([f'Ad{i+1}' for i in ads])
ax.set_xlabel('Ad'); ax.set_ylabel('CTR estimate %')
ax.set_title('CTR Estimation: MLE vs Bayesian vs Hierarchical', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('stats_03_hierarchical.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved stats_03_hierarchical.png')

print('\nKEY TAKEAWAYS for Bayesian Inference:')
print('1. Posterior = prior x likelihood / evidence (always normalize)')
print('2. Conjugate priors give analytical posteriors: Beta-Binomial, Gaussian-Gaussian')
print('3. MAP = penalized MLE; Gaussian prior = L2, Laplace prior = L1')
print('4. Full Bayes > MAP when you need uncertainty, not just a point estimate')
print('5. Hierarchical models share information across groups: lower MSE')
print('6. Prior becomes irrelevant as n -> infinity (likelihood dominates)')
print('7. Credible interval = direct probability statement; CI = sampling procedure')

print('\nEXERCISES:')
print('1. Try a Beta(10,90) prior (strong 10% belief) on the coin flip. How many flips to override it?')
print('2. Implement LASSO (Laplace prior) MAP and compare weight sparsity to Ridge')
print('3. Apply the hierarchical model when one ad has 0 clicks -- does it handle it gracefully?')